# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields
print("Available record sets (by @id):")
record_set_ids = []
for rs in metadata.record_sets:
    print(f"- {rs['@id']}: {rs['name'] if 'name' in rs else ''}")
    record_set_ids.append(rs['@id'])
    # List fields within the record set
    if 'fields' in rs:
        print("    Fields:")
        for field in rs['fields']:
            print(f"      - {field['@id']}: {field['name'] if 'name' in field else ''} (dataType: {field.get('dataType', 'N/A')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, select the first available record set
if record_set_ids:
    selected_record_set = record_set_ids[0]
    print(f"Using record set: {selected_record_set}")
else:
    raise Exception("No record sets found in the metadata.")

# Prepare dictionary of DataFrames for each record set
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} rows from record set '{rs_id}'.")
    except Exception as e:
        print(f"Could not load records for '{rs_id}': {e}")

# Show columns for the selected DataFrame
print(f"Columns in record set '{selected_record_set}':")
if selected_record_set in dataframes:
    print(dataframes[selected_record_set].columns.tolist())
    display(dataframes[selected_record_set].head())
else:
    print(f"No data loaded for record set '{selected_record_set}'.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field from the current DataFrame
# Replace '<numeric_field_id>' and '<group_field_id>' with actual @id (or column names) from the overview above as needed.
# For demonstration, we'll attempt to choose a numeric field dynamically:

import numpy as np
df = dataframes[selected_record_set]

numeric_field = None
for col in df.columns:
    try:
        if np.issubdtype(df[col].dropna().astype(float).dtype, np.number):
            numeric_field = col
            break
    except Exception:
        continue

if numeric_field:
    print(f"Found numeric field: {numeric_field}")
    threshold = df[numeric_field].dropna().quantile(0.75)  # Use 75th percentile as example threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (75th percentile):")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical or text field
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].dtype == object and df[col].nunique() > 1 and df[col].nunique() < len(df)/2:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
    else:
        print('No suitable categorical/text field found for grouping.')
else:
    print('No numeric fields identified in this record set for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # If group_field exists, show boxplot
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we successfully loaded and explored the FAIR^2 mass spectrometry dataset using the Croissant schema and `mlcroissant` library.
* We programmatically discovered available record sets and fields by their `@id`s, loaded records into DataFrames, and applied basic analysis (filtering, normalization, grouping).
* The dataset offers a rich table of protein parameters suitable for custom workflows in biomedical informatics, such as biomarker discovery or comparative studies.
* For more advanced analysis, further parsing and domain-specific computation on relevant fields can be done using the Croissant schema metadata and linked files.